# Synthesize Criteo data using the DP-GAN approach of Ponte. et al.

Note that we use the following python modules:

- tensorflow==2.15.0
- keras==2.15.0
- tensorflow-estimator==2.15.0
- tensorflow-privacy==0.9.0
- numpy==1.26.4
- pandas==2.2.2
- scikit-learn==1.4.2
- scipy==1.11.4
- absl-py==1.4.0

The following statments can be used to install the required python modules:

```bash
pip install tensorflow==2.15.0
pip install keras==2.15.0
pip install tensorflow-estimator==2.15.0
pip install tensorflow-privacy==0.9.0
pip install numpy==1.26.4
pip install pandas==2.2.2
pip install scikit-learn==1.4.2
pip install scipy==1.11.4
pip install absl-py==1.4.0
```

Perform a quick check that `tensorflow`, `keras` and `tensorflow_privacy` are installed and importable. Also check versions of `NumPy`, `Pandas`, and `Scikit-learn`.

In [5]:
# sanity check the environment
import tensorflow as tf, keras, numpy as np, pandas as pd, sklearn
from tensorflow_privacy.privacy.optimizers.dp_optimizer_keras import DPKerasAdamOptimizer

print("TF:", tf.__version__)              # 2.15.0
print("Keras:", keras.__version__)        # 2.15.0
print("NumPy:", np.__version__)           # 1.26.4
print("Pandas:", pd.__version__)          # 2.2.2
print("Sklearn:", sklearn.__version__)    # 1.4.2
_ = DPKerasAdamOptimizer(l2_norm_clip=1.0, noise_multiplier=0.5,
                         num_microbatches=1, learning_rate=1e-3)
print("DP optimizer OK")

TF: 2.15.0
Keras: 2.15.0
NumPy: 1.26.4
Pandas: 2.2.2
Sklearn: 1.4.2
DP optimizer OK


Import required packages.

In [6]:
import math
import numpy as np
import statistics
from sklearn import metrics
from functools import partial
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
import argparse
import keras
from tensorflow.keras import backend as K
from sklearn.linear_model import LinearRegression
import sys
import matplotlib.pyplot as plt
from tensorflow.keras.optimizers import Adam
import pandas as pd
import io
from keras.models import load_model
import time
from scipy.stats import pearsonr
from keras.layers import Input, Dense, Reshape, Flatten, Dropout, multiply, GaussianNoise
from keras.layers import BatchNormalization, Activation, Embedding, ZeroPadding2D
from keras.layers import MaxPooling2D, LeakyReLU
from keras.layers import UpSampling2D, Conv2D, Conv1D
from keras.models import Sequential, Model
from keras import losses
import keras.backend as K
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KernelDensity
import os
from sklearn.model_selection import train_test_split
import random
from keras.models import load_model
from absl import app
from absl import flags
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function
import logging
from tensorflow_privacy.privacy.analysis import compute_dp_sgd_privacy
from tensorflow_privacy.privacy.optimizers.dp_optimizer_keras import DPKerasSGDOptimizer, DPKerasAdamOptimizer
from tensorflow_privacy.privacy.analysis.compute_dp_sgd_privacy_lib import compute_dp_sgd_privacy
from sklearn.preprocessing import MinMaxScaler

Import Criteo data (small version is for testing, results are based on 'full' version).

In [7]:
train_data = pd.read_csv("../../Data/Criteo/cleaned_criteo_train_os.gz",
                         compression='gzip', 
                         sep='\,',
                         header=0,
                         engine='python')
data_set = "train_os"

View confidential data to synthesize.

In [8]:
train_data

,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
0,24.352730,10.059654,9.002550,4.679882,10.280525,4.115453,-9.404879,4.833815,3.892853,13.190056,5.300375,-0.168679,1,0,0,0
1,20.027816,10.059654,8.653796,3.359763,14.339694,4.115453,-9.864621,4.833815,3.781017,35.805995,5.916066,-0.473542,0,1,1,0
2,21.787905,10.059654,8.440707,4.679882,10.280525,4.115453,-8.493011,4.833815,3.842828,29.214161,5.300375,-0.168679,1,0,0,0
3,13.954588,10.059654,8.214851,-1.072603,10.280525,4.115453,-11.495164,4.833815,3.863287,33.213193,5.300375,-0.168679,1,1,1,1
4,23.970150,10.059654,8.214383,4.679882,10.280525,3.013064,-11.398781,10.591158,3.971858,13.190056,5.300375,-0.168679,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61156,24.941363,10.059654,8.214383,4.679882,10.280525,4.115453,-2.411115,4.833815,3.971858,13.190056,5.300375,-0.168679,1,0,0,0
61157,23.106934,10.059654,8.214383,4.679882,10.280525,4.115453,-1.288207,4.833815,3.971858,13.190056,5.300375,-0.168679,1,0,0,0
61158,25.975578,10.059654,8.475106,4.679882,11.561050,4.115453,-2.411115,4.833815,3.835851,23.570168,6.034478,-0.168679,1,1,1,0
61159,23.131460,10.059654,8.634955,4.679882,10.280525,4.115453,-5.576414,4.833815,3.880455,16.226044,5.300375,-0.168679,1,1,1,0


Define a class for estimating a differentially private GAN.

In [9]:
"""# GANs with differential privacy"""
class GAN():
    def __init__(self, privacy):
      self.img_rows = 1
      self.img_cols = 16
      self.img_shape = (self.img_cols,)
      self.latent_dim = (16)
      lr = 0.001

      optimizer = keras.optimizers.Adam()
      self.discriminator = self.build_discriminator()
      self.discriminator.compile(loss='binary_crossentropy',
                                 optimizer=optimizer,
                                 metrics=['accuracy'])
      if privacy == True:
        # print(noise_multiplier)
        # print("using differential privacy")
        # Build and compile the discriminator
        self.discriminator = self.build_discriminator()
        self.discriminator.compile(optimizer=DPKerasAdamOptimizer(
            l2_norm_clip=4,
            noise_multiplier=noise_multiplier,
            num_microbatches=num_microbatches,
            learning_rate=lr),
            loss= tf.keras.losses.BinaryCrossentropy(from_logits=True, reduction=tf.losses.Reduction.NONE), metrics=['accuracy'])

      # Build the generator
      self.generator = self.build_generator()

      # The generator takes noise as input and generates imgs
      z = Input(shape=(self.latent_dim,))
      img = self.generator(z)

      # For the combined model we will only train the generator
      self.discriminator.trainable = False

      # The discriminator takes generated images as input and determines validity
      valid = self.discriminator(img)

      # The combined model  (stacked generator and discriminator)
      # Trains the generator to fool the discriminator
      self.combined = Model(z, valid)
      self.combined.compile(loss='binary_crossentropy', optimizer= optimizer)


    def build_generator(self):
      model = Sequential()
      model.add(Dense(self.latent_dim, input_dim=self.latent_dim))
      model.add(LeakyReLU(alpha=0.2))
      #model.add(BatchNormalization())
      model.add(Dense(64, input_shape=self.img_shape))
      model.add(LeakyReLU(alpha=0.2))
      #model.add(BatchNormalization())
      model.add(Dense(self.latent_dim))
      model.add(Activation("tanh"))

      #model.summary()

      noise = Input(shape=(self.latent_dim,))
      img = model(noise)
      return Model(noise, img)

    def build_discriminator(self):

        model = Sequential()

        model.add(Dense(64, input_shape=self.img_shape))
        model.add(LeakyReLU(alpha=0.2))
        model.add(Dense(1, activation='sigmoid'))

        #model.summary()

        img = Input(shape=self.img_shape)
        validity = model(img)

        return Model(img, validity)

    def train(self, data, iterations, batch_size, sample_interval, model_name, generator_losses = [], discriminator_acc = [], correlations = [], accuracy = [], MAPD_collect = [],MSE_collect = [], MAE_collect = []):
      # Adversarial ground truths
      valid = np.ones((batch_size, 1))
      fake = np.zeros((batch_size, 1))
      corr = 0
      MAPD = 0
      MSE = 0
      MAE = 0
      #fake += 0.05 * np.random.random(fake.shape)
      #valid += 0.05 * np.random.random(valid.shape)

      for epoch in range(iterations):

            # ---------------------
            #  Train Discriminator
            # ---------------------

            # Select a random batch of images
            idx = np.random.randint(0, data.shape[0], batch_size)
            imgs = data[idx]

            noise = np.random.normal(0, 1, (batch_size, self.latent_dim))

            # Generate a batch of new images
            gen_imgs = self.generator.predict(noise, verbose = False)

            # Train the discriminator
            d_loss_real = self.discriminator.train_on_batch(imgs, valid)
            d_loss_fake = self.discriminator.train_on_batch(gen_imgs, fake)
            d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

            # ---------------------
            #  Train Generator
            # ---------------------
            # Train the generator (to have the discriminator label samples as valid)

            noise = np.random.normal(0, 1, (batch_size, self.latent_dim))
            g_loss = self.combined.train_on_batch(noise, valid)

            if (epoch % 100) == 0:
              print("%d [D loss: %f, acc.: %.2f%%] [G loss: %f]" % (epoch, d_loss[0], 100*d_loss[1], g_loss))

      self.generator.save(model_name)

Set number of samples (total number of observations). Determine epochs as a function of batch size, which we leave fixed at 100 (same as Ponte et al.), and scale iterations to have `epochs = 10`.

In [10]:
# number of samples in the data set
samples = int(train_data.shape[0])

# setting epsilon
N = len(train_data)
batch_size = 100

In [11]:
N

61161

In [12]:
### change for different data sizes
iterations = 10000
epochs = iterations/(N/batch_size) # should be 10
num_microbatches = batch_size # see validation section paper.

# the noise_multiplier is not directly passed to the GAN, but the GAN code reads it from the global environment
l2_norm_clip = 4 # see paper in validation section.
delta = 1/N # should be 1/N

In [13]:
epochs

16.350288582593482

In [94]:
# define a list of different noise multipliers to use for synthesis
noise_multipliers = [0.45134, 0.6679, 0.9988, 1.433, 10.3]

Choose noise multipliers that map to $\epsilon = 13, 3, 1, 0.5, 0.05$. The `tensorflow-privacy` package has deprecated the use of the `compute_dp_sgd_privacy` function, replacing it with `compute_dp_sgd_privacy_statement` which properly accounts for doubling sensitivity due to microbatching and does not assume Poisson subsampling. However, we use the existing methods from Ponte et al. for consistency, and note that the theoretical epsilon is higher than what is reported.

In [95]:
# calculate the theoretical bound of epsilon
[np.round(compute_dp_sgd_privacy(n = N, 
                                 batch_size = batch_size,
                                 epochs = epochs,
                                 noise_multiplier = x,
                                 delta = delta)[0], 3) for x in noise_multipliers] 

[13.0, 3.0, 1.0, 0.5, 0.05]

---

In [96]:
# import warnings
# warnings.filterwarnings('ignore')

# import os
# import logging
# import tensorflow as tf
# from absl import logging as absl_logging

# # Suppress low-level TF C++ logs (0=all, 1=INFO, 2=WARNING, 3=ERROR)
# os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# # Suppress Python-level TF warnings
# tf.get_logger().setLevel(logging.ERROR)
# logging.getLogger("tensorflow").setLevel(logging.ERROR)
# absl_logging.set_verbosity(absl_logging.ERROR)

all_synthetic_datasets = {}

"""iteraties en batch size hetzelfde houden."""
random.seed(1)
np.random.seed(1)
tf.random.set_seed(1)

start_time = time.time()

# scale data for GAN training
scaler0 = MinMaxScaler(feature_range = (-1, 1))
scaler0 = scaler0.fit(train_data)
train_GAN_real = scaler0.transform(train_data)
train_GAN_real = pd.DataFrame(train_GAN_real)

# we vary the noise multipliers here, train a GAN and generate multiple synthetic data sets for each noise multiplier
for iter, noise_multiplier in enumerate(noise_multipliers): 
  random.seed(iter)
  np.random.seed(iter)
  tf.random.set_seed(iter)

  # train GAN on train data
  gan_train = GAN(privacy = True)
  gan_train.train(data = np.array(train_GAN_real), iterations=iterations, batch_size=batch_size, sample_interval=((iterations-1)/10), model_name = "train_1.h5")

  print('Model trained.')

  # list to store synthetic data sets
  synthetic_datasets = []
  # load model
  generator = load_model('train_1.h5')
  # for 20 iterations
  for data_num in range(20):
    # set random seeds
    random.seed(data_num)
    np.random.seed(data_num)
    tf.random.set_seed(data_num)
    # generate a synthetic data set
    synthetic_datasets.append(generator.predict(np.random.normal(0, 1, (samples, 16)), verbose = False))
    print('Created ' + str(data_num+1) + "/" + "20 synthetic data sets.")
  # invert the min-max transformation
  synthetic_datasets = [scaler0.inverse_transform(X) for X in synthetic_datasets]
  # reshape to correct size
  synthetic_datasets = [pd.DataFrame(X.reshape(samples, 16)) for X in synthetic_datasets]
  # replace column names and round categorical variables
  for X in synthetic_datasets:
    # replace column names
    X.columns = train_data.columns.values
    ####################################################
    # round the values of categorical variables, as done by Ponte et al.
    ####################################################
    X['treatment'] = X['treatment'].round()
    X['conversion'] = X['conversion'].round()
    X['visit'] = X['visit'].round()
    X['exposure'] = X['exposure'].round()

  all_synthetic_datasets[str(noise_multiplier)] = synthetic_datasets

C:\Users\Cam\anaconda3\envs\dp-gan\Lib\site-packages\keras\src\backend.py:5818: UserWarning: "`binary_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Sigmoid activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(



0 [D loss: 0.694357, acc.: 57.50%] [G loss: 0.809698]
100 [D loss: 0.753197, acc.: 43.00%] [G loss: 0.536334]
200 [D loss: 0.577947, acc.: 71.50%] [G loss: 0.689984]
300 [D loss: 0.657731, acc.: 57.00%] [G loss: 0.750413]
400 [D loss: 0.581432, acc.: 67.00%] [G loss: 0.865659]
500 [D loss: 0.737575, acc.: 52.50%] [G loss: 0.779068]
600 [D loss: 0.526279, acc.: 91.00%] [G loss: 0.887637]
700 [D loss: 0.570260, acc.: 79.50%] [G loss: 0.889094]
800 [D loss: 0.602085, acc.: 75.50%] [G loss: 0.964508]
900 [D loss: 0.657861, acc.: 58.00%] [G loss: 0.713092]
1000 [D loss: 0.795495, acc.: 36.50%] [G loss: 0.719278]
1100 [D loss: 0.539508, acc.: 84.00%] [G loss: 1.013139]
1200 [D loss: 0.583538, acc.: 82.50%] [G loss: 0.877057]
1300 [D loss: 0.710585, acc.: 49.00%] [G loss: 0.670189]
1400 [D loss: 0.564647, acc.: 82.00%] [G loss: 0.885359]
1500 [D loss: 0.681369, acc.: 55.00%] [G loss: 0.750973]
1600 [D loss: 0.592079, acc.: 78.50%] [G loss: 0.859170]
1700 [D loss: 0.614053, acc.: 80.50%] [G lo

C:\Users\Cam\anaconda3\envs\dp-gan\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Model trained.


Created 1/20 synthetic data sets.
Created 2/20 synthetic data sets.
Created 3/20 synthetic data sets.
Created 4/20 synthetic data sets.
Created 5/20 synthetic data sets.
Created 6/20 synthetic data sets.
Created 7/20 synthetic data sets.
Created 8/20 synthetic data sets.
Created 9/20 synthetic data sets.
Created 10/20 synthetic data sets.
Created 11/20 synthetic data sets.
Created 12/20 synthetic data sets.
Created 13/20 synthetic data sets.
Created 14/20 synthetic data sets.
Created 15/20 synthetic data sets.
Created 16/20 synthetic data sets.
Created 17/20 synthetic data sets.
Created 18/20 synthetic data sets.
Created 19/20 synthetic data sets.
Created 20/20 synthetic data sets.


...

C:\Users\Cam\anaconda3\envs\dp-gan\Lib\site-packages\keras\src\backend.py:5818: UserWarning: "`binary_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Sigmoid activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


0 [D loss: 0.559047, acc.: 54.50%] [G loss: 0.557087]
100 [D loss: 0.685152, acc.: 50.00%] [G loss: 0.516147]
200 [D loss: 0.655370, acc.: 54.50%] [G loss: 0.681889]
300 [D loss: 0.742275, acc.: 4.50%] [G loss: 0.672767]
400 [D loss: 0.673833, acc.: 74.50%] [G loss: 0.737377]
500 [D loss: 0.599580, acc.: 92.00%] [G loss: 0.783466]
600 [D loss: 0.774667, acc.: 42.50%] [G loss: 0.594016]
700 [D loss: 0.625942, acc.: 75.50%] [G loss: 0.818032]
800 [D loss: 0.592161, acc.: 85.50%] [G loss: 0.824624]
900 [D loss: 0.664923, acc.: 64.00%] [G loss: 0.747713]
1000 [D loss: 0.633250, acc.: 64.00%] [G loss: 0.710024]
1100 [D loss: 0.690363, acc.: 42.50%] [G loss: 0.698587]
1200 [D loss: 0.677285, acc.: 52.50%] [G loss: 0.788352]
1300 [D loss: 0.727393, acc.: 32.50%] [G loss: 0.605154]
1400 [D loss: 0.551189, acc.: 94.50%] [G loss: 0.915608]
1500 [D loss: 0.579681, acc.: 68.50%] [G loss: 0.675272]
1600 [D loss: 0.787817, acc.: 35.00%] [G loss: 0.519883]
1700 [D loss: 0.717019, acc.: 31.00%] [G los

C:\Users\Cam\anaconda3\envs\dp-gan\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Model trained.


Created 1/20 synthetic data sets.
Created 2/20 synthetic data sets.
Created 3/20 synthetic data sets.
Created 4/20 synthetic data sets.
Created 5/20 synthetic data sets.
Created 6/20 synthetic data sets.
Created 7/20 synthetic data sets.
Created 8/20 synthetic data sets.
Created 9/20 synthetic data sets.
Created 10/20 synthetic data sets.
Created 11/20 synthetic data sets.
Created 12/20 synthetic data sets.
Created 13/20 synthetic data sets.
Created 14/20 synthetic data sets.
Created 15/20 synthetic data sets.
Created 16/20 synthetic data sets.
Created 17/20 synthetic data sets.
Created 18/20 synthetic data sets.
Created 19/20 synthetic data sets.
Created 20/20 synthetic data sets.


Save synthetic data sets.

In [97]:
synthetic_data_path = "../../Data/Criteo/"
epsilons = ["13", "3", "1", "05", "005"]

for e, item in enumerate(all_synthetic_datasets.items()):
    sXs = item[1]
    if not os.path.exists(synthetic_data_path):
        os.makedirs(synthetic_data_path)
    for i, X in enumerate(sXs):
        X.to_csv(synthetic_data_path + "dpgan_" + epsilons[e] + "_" + str(i) + "_" + data_set + ".csv", index=False)

End of file.